# Capstone — Search Intelligence Content Refresh Prioritization

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/capstone.ipynb?flush_cache=true)

This notebook contains the complete capstone machine learning pipeline, executing feature engineering, client-holdout split validation, model training, precision evaluation, error analysis, and artifact generation.

## 1. Question

*The research question and the decision it supports.*

In [ ]:
print("RESEARCH QUESTION:")
print("Can machine learning classification models predict 90-day search performance decline risk")
print("and prioritize content refresh candidates under a strict client-holdout validation split?")
print()
print("DECISION SUPPORTED:")
print("Helps SEO content strategists allocate monthly editorial refresh capacity to high-value pages")
print("at risk of decay, replacing manual intuition with a leak-free ranked queue.")

## 2. Data

*Which release, which tables, date windows, what you excluded and why. Public-safe.*

In [ ]:
import os, sys
import numpy as np
import pandas as pd

data_path = 'data/raw/content_refresh_anonymized.csv'
if not os.path.exists(data_path):
    data_path = 'https://raw.githubusercontent.com/Srujanmp1366/flyrank-internship/main/data/raw/content_refresh_anonymized.csv'

df = pd.read_csv(data_path)

# Clean numeric fields by filling NaNs
for col in ['impressions_90d', 'clicks_90d', 'sessions_90d', 'days_since_last_update', 'avg_position',
            'word_count', 'ctr', 'engagement_rate', 'scroll_rate', 'content_age_days', 'search_volume', 'competition']:
    df[col] = df[col].fillna(0)

df['content_type'] = df['content_type'].fillna('unknown')
df['main_intent'] = df['main_intent'].fillna('unknown')
df['trend_direction'] = df['trend_direction'].fillna('unknown')
df['is_declining_label'] = (df['trend_direction'].astype(str).str.lower() == 'down').astype(int)

print(f"Dataset Source : {data_path}")
print(f"Total Rows     : {len(df):,}")
print(f"Total Columns  : {len(df.columns)}")
print(f"Unique Clients : {df['client_id'].nunique()}")
print(f"Base Rate      : {df['is_declining_label'].mean():.3f} ({df['is_declining_label'].mean()*100:.1f}% declining)")
print()
print("EXCLUDED COLUMNS & REASONS:")
print("- trend_direction, trend_pct : Label-derived features (Leakage Risk)")
print("- content_id, client_id     : Pseudonymous IDs (Group/Join Only)")

## 3. Methodology

*Assumptions, features, label definition, baseline, validation design, leakage checks.*

In [ ]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import roc_auc_score, average_precision_score

numeric_cols = ['impressions_90d', 'clicks_90d', 'sessions_90d', 'ctr', 'avg_position', 
                'engagement_rate', 'scroll_rate', 'content_age_days', 'days_since_last_update',
                'word_count', 'search_volume', 'competition']
categorical_cols = ['content_type', 'main_intent']

preprocessor = ColumnTransformer(
    transformers=[
        ('num', StandardScaler(), numeric_cols),
        ('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), categorical_cols)
    ]
)

# Client Holdout Split
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
X = df[numeric_cols + categorical_cols]
y = df['is_declining_label']
groups = df['client_id']

train_idx, test_idx = next(gss.split(X, y, groups=groups))
X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train Rows : {len(X_train):,} ({df.iloc[train_idx]['client_id'].nunique()} Clients)")
print(f"Test Rows  : {len(X_test):,} ({df.iloc[test_idx]['client_id'].nunique()} Clients)")
print("Validation Design: Grouped Client Holdout (Zero Leakage Across Domains) [PASS]")

## 4. Results (vs baseline)

*Model vs baseline on the same split. The honest table.*

In [ ]:
def precision_at_k(scores, labels, k=50):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

# Baseline Score on Test Set
test_df = df.iloc[test_idx]
impr_rank = test_df['impressions_90d'].rank(pct=True)
stale_rank = test_df['days_since_last_update'].rank(pct=True)
pos_norm = (test_df['avg_position'].clip(1, 50) - 1) / 49.0
pos_opp = (1 - pos_norm) * impr_rank * (test_df['avg_position'] > 0).astype(int)
depth_gap = (1 - test_df['word_count'].rank(pct=True)) * impr_rank
baseline_test_scores = (0.40 * impr_rank + 0.30 * stale_rank + 0.25 * pos_opp + 0.05 * depth_gap).clip(0, 1)

p50_base = precision_at_k(baseline_test_scores, y_test, 50)

# Train Random Forest
model_rf = Pipeline([
    ('preprocessor', preprocessor),
    ('classifier', RandomForestClassifier(class_weight='balanced_subsample', max_depth=10, min_samples_leaf=25, n_estimators=200, random_state=42))
])
model_rf.fit(X_train, y_train)
rf_probs = model_rf.predict_proba(X_test)[:, 1]
p50_rf = precision_at_k(rf_probs, y_test, 50)
auc_rf = roc_auc_score(y_test, rf_probs)
pr_auc_rf = average_precision_score(y_test, rf_probs)

print("CAPSTONE MODEL RESULTS SUMMARY (HOLDOUT TEST SET):")
print("=" * 90)
print(f"Baseline Rule Precision@50  : {p50_base:.3f}")
print(f"Random Forest Precision@50 : {p50_rf:.3f}")
print(f"Random Forest ROC-AUC      : {auc_rf:.3f}")
print(f"Random Forest PR-AUC       : {pr_auc_rf:.3f}")
print(f"Precision Lift             : {p50_rf / p50_base:.2f}x lift over baseline rule")

## 5. Limitations

*What this work cannot claim.*

In [ ]:
print("LIMITATIONS & PUBLIC SAFETY BOUNDS:")
print("-" * 80)
print("1. NO CAUSAL PROOF : Model finds correlation in historical 90-day data; does not prove causality.")
print("2. NO GOOGLE ALGORITHM PREDICTION : Evaluates our client performance, not internal search engine mechanics.")
print("3. MATURE CONTENT ONLY : Requires >= 90 days of search history; not valid for brand-new pages.")

## 6. Ranked recommendations

*The action playbook output — the paper's recommendations section.*

In [ ]:
print("RECOMMENDED EDITORIAL WORKFLOW:")
print("1. Route top 50 model-ranked pages into monthly editorial review.")
print("2. Apply specific reason-code playbooks (expansion, title rewrite, staleness refresh).")
print("3. Require mandatory human review checklist before publishing.")

## 7. Artifacts the paper embeds

*Generate/collect the charts and tables your deployed page will show.*

In [ ]:
import matplotlib.pyplot as plt

os.makedirs('work/outputs/figures', exist_ok=True)

# Generate Capstone Precision@K Comparison Chart
ks = [10, 20, 50, 100]
base_p = [precision_at_k(baseline_test_scores, y_test, k) for k in ks]
rf_p = [precision_at_k(rf_probs, y_test, k) for k in ks]

plt.figure(figsize=(8, 4.5))
plt.plot(ks, base_p, marker='o', label='Baseline Rule', linestyle='--')
plt.plot(ks, rf_p, marker='s', label='Random Forest Classifier', color='green')
plt.title('Precision@K Comparison on Unseen Client Holdout')
plt.xlabel('Top K Pages Evaluated')
plt.ylabel('Precision@K (Decline Rate)')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig('work/outputs/figures/capstone_precision_k.png', dpi=300)
plt.close()

print("Capstone Precision Chart Saved to: work/outputs/figures/capstone_precision_k.png [PASS]")

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.